# 04 — Linearity diagnostics with power analysis

anchor-op ships two linearity diagnostics: `linearity_check` (bin-split by κ, compares operators
across bins) and `held_out_prediction_check` (5-fold CV on the identity `J·S_g = −U_g`). At
published Perturb-seq scale, both diagnostics are **noise-limited** — their observed values are
often indistinguishable between a linear model and a moderately-nonlinear one.

**Do not interpret observed diagnostic values against fixed thresholds without a matched-scale
positive control.** This tutorial shows exactly how to do that check.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import anchorop as ao
from anchorop.identifiability import regularized_pseudoinverse
rng = np.random.default_rng(0)

## Basic usage

Given a `MeasuredOperator`, both diagnostics take one line each.

In [ ]:
# Build a small synthetic measurement (see tutorial 01)
def synthetic_measurement(J, n_guides, kappa_range, noise):
    d = J.shape[0]
    Wd = rng.normal(size=(d, n_guides)); Wd /= np.linalg.norm(Wd, axis=0, keepdims=True)
    kappa = rng.uniform(kappa_range[0], kappa_range[1], size=n_guides)
    U = -kappa[None, :] * Wd
    S = -np.linalg.solve(J, U) + noise * rng.normal(size=(d, n_guides))
    names = [f"g{i}" for i in range(n_guides)]
    effs = {n: float(kappa[i]) for i, n in enumerate(names)}
    return ao.measure_from_sensitivity(S, U, guide_names=names, guide_efficiencies=effs,
                                        reg="tsvd", reg_param="path", rank_tol=1e-2)

# A "small clean" test case where both diagnostics have power
d = 6
J = rng.normal(size=(d, d)) - 2 * np.eye(d)
m_clean = synthetic_measurement(J, n_guides=60, kappa_range=(0.05, 0.95), noise=0.01)

lin = ao.linearity_check(m_clean, threshold=0.25, n_null=100, null_seed=42)
rho = ao.held_out_prediction_check(m_clean, n_folds=5, seed=0, n_permutation_null=30)
print(f"Clean small case (d=6, n=60, wide κ, low noise):")
print(f"  rel_diff = {lin.relative_difference:.3f}  (< 0.25 threshold: {lin.passed})")
print(f"  null_median = {lin.null_median:.3f}")
print(f"  held-out ρ = {rho.rho_pooled:.3f}  (null median = {rho.null_median:.3f})")

## The power-analysis methodology (paper §3.5)

At realistic Perturb-seq scale (d≈30, n≈200 guides, narrow κ, high per-guide noise), the same
diagnostics are noise-limited: they cannot distinguish linear from moderately nonlinear. The
correct workflow is to **compute a matched-scale positive control** and compare.

Here's the recipe for any measurement:

In [ ]:
def matched_scale_positive_control(measurement, noise_sigma, n_reps=10, seed_base=1000):
    """Draw a synthetic LINEAR ground truth at the same (d, n_guides, U, κ) as `measurement`,
    add per-entry Gaussian noise of scale `noise_sigma`, and return the mean/std of both
    diagnostics across n_reps replicates."""
    U = measurement.U
    kappa = np.array([measurement.report.guide_efficiencies[g] for g in measurement.guide_names])
    d, n_guides = measurement.S.shape
    rank_tol = measurement.report.rank_tol

    rel_diffs, rhos = [], []
    for rep in range(n_reps):
        rng_j = np.random.default_rng(seed_base + rep)
        J_true = (rng_j.normal(size=(d, d)) / np.sqrt(d)) - 1.5 * np.eye(d)
        S_syn = -np.linalg.solve(J_true, U) + noise_sigma * rng_j.normal(size=(d, n_guides))
        m_syn = ao.measure_from_sensitivity(
            S_syn, U,
            guide_names=list(measurement.guide_names),
            guide_efficiencies=dict(measurement.report.guide_efficiencies),
            reg="tsvd", reg_param="path", rank_tol=rank_tol,
        )
        rel_diffs.append(ao.linearity_check(m_syn, n_null=50, null_seed=42).relative_difference)
        rhos.append(ao.held_out_prediction_check(m_syn, n_folds=5, seed=0).rho_pooled)
    return {
        "rel_diff_mean": np.mean(rel_diffs), "rel_diff_std": np.std(rel_diffs),
        "rho_mean": np.mean(rhos), "rho_std": np.std(rhos),
    }

## Demo: what would Replogle-scale data look like?

We build a synthetic measurement at Replogle-like scale (d=30, n=200 guides, narrow κ) with
realistic per-guide Δz noise (σ = 0.266, measured on Replogle K562 essential; see paper §3.5).
Then we run both diagnostics AND the matched-scale positive control.

In [ ]:
d, n = 30, 200
J_realscale = (rng.normal(size=(d, d)) / np.sqrt(d)) - 1.5 * np.eye(d)
# Add a mild nonlinearity: soft saturation, sat=1.0
Wd = rng.normal(size=(d, n)); Wd /= np.linalg.norm(Wd, axis=0, keepdims=True)
kappa = 0.05 + 0.45 * rng.uniform(size=n)   # narrow κ (Replogle-shape)
U = -kappa[None, :] * Wd
S_lin = -np.linalg.solve(J_realscale, U)
sat = 1.0
S_signal = sat * np.tanh(S_lin / sat)         # apply saturation
S = S_signal + 0.266 * rng.normal(size=(d, n))
names = [f"g{i}" for i in range(n)]; effs = {nm: float(kappa[i]) for i, nm in enumerate(names)}
m_realscale = ao.measure_from_sensitivity(S, U, guide_names=names, guide_efficiencies=effs,
                                            reg="tsvd", reg_param="path", rank_tol=1e-2)

# Observed diagnostics
lin_obs = ao.linearity_check(m_realscale, n_null=50, null_seed=42)
rho_obs = ao.held_out_prediction_check(m_realscale, n_folds=5, seed=0)
print(f"OBSERVED (mild saturation + realistic noise):")
print(f"  rel_diff = {lin_obs.relative_difference:.3f}  (well above 0.25 threshold!)")
print(f"  held-out ρ = {rho_obs.rho_pooled:.3f}  (near zero-predictor ρ=1)")

# Matched-scale positive control: what LINEAR data would give here
print("\nComputing matched-scale linear positive control (may take ~1 min)...")
pc = matched_scale_positive_control(m_realscale, noise_sigma=0.266, n_reps=5)
print(f"POSITIVE CONTROL (synthetic LINEAR at matched d, n, U, κ, σ):")
print(f"  rel_diff = {pc['rel_diff_mean']:.3f} ± {pc['rel_diff_std']:.3f}")
print(f"  held-out ρ = {pc['rho_mean']:.3f} ± {pc['rho_std']:.3f}")

print("\n→ If observed values are within a few std of the positive control,")
print("  the diagnostic cannot discriminate linear from nonlinear at this scale.")

## Interpretation guide

Three regimes:

1. **Observed ≪ positive control**: nothing sensible; check inputs.
2. **Observed ≈ positive control (within ~2σ)**: diagnostic is noise-limited at this scale.
   You cannot make a linearity claim either way from this data + these diagnostics. This is what
   the paper reports for Replogle K562 and RPE1 essential-gene screens.
3. **Observed ≫ positive control (many σ above)**: evidence of nonlinearity beyond the noise floor.
   The magnitude of the gap tells you how much nonlinearity is above the diagnostic's power.

The paper's §3.5 and §3.6 characterize the noise-floor regime in detail and derive the design
requirements (wide κ, ~50k cells/guide) for the diagnostics to gain rejection power against
moderate saturating nonlinearity.

## Next

- **05**: comparing anchor-op's measured operator against inferred operators from other tools.
- **06**: high-level `ao.analyses.*_report` shortcuts.